# 02. Case Representation & Metadata Extraction

This notebook reads the raw PDF court decisions, extracts key structured metadata using heuristic rule-based parsing, and represents them as a unified case database.

### Extracted Metadata Fields:
1. **Case ID**: Generated from the document filename.
2. **Nomor perkara**: Case/document docket number.
3. **Tanggal putusan**: Date of the court decision.
4. **Jenis perkara**: Type of legal case (e.g., Pemalsuan Surat).
5. **Pengadilan**: The court deciding the case.
6. **Pasal**: Legal articles violated (e.g., Pasal 263, 264, 266 KUHP).
7. **Nama terdakwa**: Name of the defendant.
8. **Ringkasan fakta**: Summary of case facts or description.
9. **Amar putusan**: Verdict/ruling of the judges.
10. **Jumlah kata**: Preprocessed/cleaned word count.

### Exports:
- `data/processed/cases.csv`: Structured dataset in tabular CSV format.
- `data/processed/cases.json`: Structured dataset in JSON format for the CBR system.

In [ ]:
import os
import re
import json
import logging
from pathlib import Path
import pdfplumber
import pandas as pd

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("CBR_CaseRepresentation")

In [ ]:
# Define paths
RAW_PDF_DIR = Path("../data/raw_pdf")
RAW_TEXT_DIR = Path("../data/raw_text")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_raw_pdf_text(pdf_path: Path) -> str:
    """Extracts the exact raw text from PDF using pdfplumber."""
    text_pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_pages.append(page_text)
    return "\n".join(text_pages)

In [ ]:
def clean_spaces(text: str) -> str:
    """Cleans double spaces and redundant newlines."""
    return re.sub(r'\s+', ' ', text).strip()

def parse_metadata(raw_text: str, case_id: str, cleaned_word_count: int) -> dict:
    """Heuristically extracts court decision metadata from raw text."""
    
    # 1. Nomor Perkara
    # Common pattern: "Nomor : ..." or "Putusan Nomor ..."
    no_perkara = "Tidak ditemukan"
    no_perkara_match = re.search(r'(?i)Nomor\s*:\s*([^\n]+)', raw_text)
    if not no_perkara_match:
        no_perkara_match = re.search(r'(?i)Putusan\s+Nomor\s+([^\n\/]+\/[^\n]+)', raw_text)
    if no_perkara_match:
        no_perkara = clean_spaces(no_perkara_match.group(1))
        
    # 2. Tanggal Putusan
    # Match date patterns in Indonesian: "tanggal 12 Oktober 2023" or similar
    tanggal_putusan = "Tidak ditemukan"
    tanggal_match = re.search(r'(?i)tanggal\s+(\d{1,2}\s+(?:Januari|Februari|Maret|April|Mei|Juni|Juli|Agustus|September|Oktober|November|Desember)\s+\d{4})', raw_text)
    if tanggal_match:
        tanggal_putusan = tanggal_match.group(1).strip()
        
    # 3. Pengadilan
    pengadilan = "Tidak ditemukan"
    pengadilan_match = re.search(r'(?i)(PENGADILAN\s+NEGERI\s+[A-Z\s]+|PENGADILAN\s+TINGGI\s+[A-Z\s]+|MAHKAMAH\s+AGUNG)', raw_text)
    if pengadilan_match:
        pengadilan = clean_spaces(pengadilan_match.group(1))
        
    # 4. Jenis Perkara
    # Since this is focused on document forgery, check text content or assign a default
    jenis_perkara = "Pemalsuan Surat"
    if "pemalsuan" in raw_text.lower():
        jenis_perkara = "Pemalsuan Surat"
    elif "memalsukan" in raw_text.lower():
        jenis_perkara = "Pemalsuan Surat"
        
    # 5. Pasal
    # Look for Article 263, 264, 266 etc of KUHP
    pasal_list = re.findall(r'(?i)pasal\s+\d+(?:\s+ayat\s+\(\d+\))?\s*(?:jo\.?\s*)?(?:pasal\s+\d+)*\s*(?:ke\-\d+)?\s*(?:kuhp|kitab\s+undang-undang\s+hukum\s+pidana)', raw_text)
    pasal = ", ".join(sorted(list(set([clean_spaces(p) for p in pasal_list])))) if pasal_list else "Pasal 263 KUHP"
    
    # 6. Nama Terdakwa
    nama_terdakwa = "Tidak ditemukan"
    terdakwa_match = re.search(r'(?i)(?:Nama\s+lengkap|Nama)\s*:\s*([^\n\,]+)', raw_text)
    if not terdakwa_match:
        terdakwa_match = re.search(r'(?i)Terdakwa\s*:\s*([^\n\,]+)', raw_text)
    if terdakwa_match:
        nama_terdakwa = clean_spaces(terdakwa_match.group(1))
        # Clean any trailing punctuation
        nama_terdakwa = re.sub(r'[^a-zA-Z\s\.]', '', nama_terdakwa).strip()
        
    # 7. Ringkasan Fakta
    # Extracts paragraph under Menimbang bahwa Terdakwa didakwa... or generic facts summary
    ringkasan_fakta = "Tidak ditemukan"
    fakta_match = re.search(r'(?i)Menimbang,\s+bahwa\s+Terdakwa\s+diajukan\s+ke\s+persidangan\s*(.*?)(?:Menimbang|$)', raw_text, re.DOTALL)
    if fakta_match:
        ringkasan_fakta = clean_spaces(fakta_match.group(1))
        if len(ringkasan_fakta) > 400:
            ringkasan_fakta = ringkasan_fakta[:400] + "..."
    else:
        # Fallback to the indictment section or start of document
        fakta_fallback = re.search(r'(?i)(?:dakwaan|didakwa)\s*(.*?)(?:Menimbang|$)', raw_text, re.DOTALL)
        if fakta_fallback:
            ringkasan_fakta = clean_spaces(fakta_fallback.group(1))[:400] + "..."
        else:
            ringkasan_fakta = clean_spaces(raw_text[:400]) + "..."
            
    # 8. Amar Putusan
    # Extract section starting with MENGADILI:
    amar_putusan = "Tidak ditemukan"
    amar_match = re.search(r'(?i)MENGADILI\s*:\s*(.*?)(?:Keadaan\s+yang|Demikian|Panitera|dijatuhkan|$)', raw_text, re.DOTALL)
    if amar_match:
        amar_putusan = clean_spaces(amar_match.group(1))
        if len(amar_putusan) > 400:
            amar_putusan = amar_putusan[:400] + "..."
            
    return {
        "Case ID": case_id,
        "Nomor perkara": no_perkara,
        "Tanggal putusan": tanggal_putusan,
        "Jenis perkara": jenis_perkara,
        "Pengadilan": pengadilan,
        "Pasal": pasal,
        "Nama terdakwa": nama_terdakwa,
        "Ringkasan fakta": ringkasan_fakta,
        "Amar putusan": amar_putusan,
        "Jumlah kata": cleaned_word_count
    }

In [ ]:
# Execute extraction loop
pdf_files = list(RAW_PDF_DIR.glob("*.pdf"))
cases = []

logger.info(f"Processing {len(pdf_files)} files for metadata extraction.")

for pdf_path in pdf_files:
    case_id = pdf_path.stem
    txt_path = RAW_TEXT_DIR / f"{case_id}.txt"
    
    # Get preprocessed word count
    cleaned_word_count = 0
    if txt_path.exists():
        with open(txt_path, "r", encoding="utf-8") as f:
            cleaned_word_count = len(f.read().split())
            
    try:
        logger.info(f"Extracting metadata from: {pdf_path.name}")
        raw_text = extract_raw_pdf_text(pdf_path)
        
        case_data = parse_metadata(raw_text, case_id, cleaned_word_count)
        cases.append(case_data)
        
    except Exception as e:
        logger.error(f"Failed to parse {pdf_path.name}: {e}")

In [ ]:
# Create structured datasets
df_cases = pd.DataFrame(cases)

if not df_cases.empty:
    # Save CSV
    csv_output_path = PROCESSED_DIR / "cases.csv"
    df_cases.to_csv(csv_output_path, index=False, encoding="utf-8")
    logger.info(f"Saved cases CSV to {csv_output_path.resolve()}")
    
    # Save JSON
    json_output_path = PROCESSED_DIR / "cases.json"
    df_cases.to_json(json_output_path, orient="records", indent=4, force_ascii=False)
    logger.info(f"Saved cases JSON to {json_output_path.resolve()}")
    
    # Display the cases dataframe
    display(df_cases)
else:
    logger.warning("No cases parsed. CSV and JSON files were not generated.")